<a href="https://colab.research.google.com/github/Jivinlol/QSVM/blob/main/QSVM1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install numpy pandas scikit-learn matplotlib joblib qiskit qiskit-aer qiskit-machine-learning


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.5 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import joblib

# Quantum libraries
from qiskit_aer import Aer
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

from google.colab import drive
#drive.mount('/content/drive')


print("import working properly")
DATA_CSV = "data/sensor_readings4.csv"   # your dataset path
SOIL_COL = "Soil Moisture"              # feature column (example name)
TARGET_COL = "Pump Data"                # label column (0 = off, 1 = on)
SOIL_THRESHOLD = 0.25                   # threshold to generate labels if missing
PCA_COMPONENTS = 3                     # also number of qubits
TEST_SIZE = 0.3
RANDOM_STATE = 42
save_path = '/content/drive/MyDrive/qsvc_trained_model.pkl'

if not os.path.exists(DATA_CSV):
    raise FileNotFoundError(f"File not found: {DATA_CSV}")

df = pd.read_csv(DATA_CSV)
# Select numeric columns for training
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COL]
X = df[num_cols].values
y = df[TARGET_COL].values

# =======================
# 🔀 TRAIN-TEST SPLIT
# =======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# =======================
# ⚗️ PREPROCESSING (Scaling + PCA)
# =======================
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

pca = PCA(n_components=min(PCA_COMPONENTS, X_train_s.shape[1]), random_state=RANDOM_STATE).fit(X_train_s)
X_train_p = pca.transform(X_train_s)
X_test_p = pca.transform(X_test_s)

print(f"PCA Explained Variance: {pca.explained_variance_ratio_.sum():.3f}")

# =======================
# 💡 CLASSICAL BASELINE (RBF SVM)
# =======================
clf = SVC(kernel='rbf', class_weight='balanced', random_state=RANDOM_STATE)
param_grid = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']}
gs = GridSearchCV(clf, param_grid, cv=3, scoring='f1', n_jobs=-1)
gs.fit(X_train_p, y_train)
y_pred_cl = gs.best_estimator_.predict(X_test_p)
print(X_test[0])
print(X_test_s[0])
print(X_test_p[0])
print(y_pred_cl[0])

print("\n===== Classical SVM Performance =====")
print(classification_report(y_test, y_pred_cl))

# =======================
# ⚛️ QUANTUM KERNEL SVM
# =======================
print("\nSetting up Quantum SVM...")
X_train_p =  X_train_p * np.pi
X_test_p = X_test_p * np.pi

# Quantum Feature Map
feature_map = ZZFeatureMap(
    feature_dimension=pca.n_components_,
    reps=1,
    entanglement="linear"
)

# Directly initialize FidelityQuantumKernel (no backend/sampler args)
qkernel = FidelityQuantumKernel(feature_map=feature_map)
k_matrix = qkernel.evaluate(X_train_p)
print(f"Mean kernel value: {np.mean(k_matrix)}")
classifier = QSVC(quantum_kernel=qkernel)
classifier.fit(X_train_p, y_train)
y_pred_q = classifier.predict(X_test_p)



print("\n===== Quantum SVM Performance =====")
print(classification_report(y_test, y_pred_q))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_test, y_pred_q))

#joblib.dump(classifier, save_path)

print(f"✅ Model saved successfully to: {save_path}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import joblib

# Quantum libraries
from qiskit_aer import Aer
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

from google.colab import drive
drive.mount('/content/drive')


print("import working properly")
DATA_CSV = "data/sensor_readings4.csv"   # your dataset path
SOIL_COL = "Soil Moisture"              # feature column (example name)
TARGET_COL = "Pump Data"                # label column (0 = off, 1 = on)
SOIL_THRESHOLD = 0.25                   # threshold to generate labels if missing
PCA_COMPONENTS = 3                     # also number of qubits
TEST_SIZE = 0.3
RANDOM_STATE = 42
save_path = '/content/drive/MyDrive/qsvc_trained_model.pkl'

if not os.path.exists(DATA_CSV):
    raise FileNotFoundError(f"File not found: {DATA_CSV}")

df = pd.read_csv(DATA_CSV)
# Select numeric columns for training
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COL]
X = df[num_cols].values
y = df[TARGET_COL].values

# =======================
# 🔀 TRAIN-TEST SPLIT
# =======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# =======================
# ⚗️ PREPROCESSING (Scaling + PCA)
# =======================
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

pca = PCA(n_components=min(PCA_COMPONENTS, X_train_s.shape[1]), random_state=RANDOM_STATE).fit(X_train_s)
X_train_p = pca.transform(X_train_s)
X_test_p = pca.transform(X_test_s)

print(f"PCA Explained Variance: {pca.explained_variance_ratio_.sum():.3f}")

# =======================
# 💡 CLASSICAL BASELINE (RBF SVM)
# =======================
clf = SVC(kernel='rbf', class_weight='balanced', random_state=RANDOM_STATE)
param_grid = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']}
gs = GridSearchCV(clf, param_grid, cv=3, scoring='f1', n_jobs=-1)
gs.fit(X_train_p, y_train)
y_pred_cl = gs.best_estimator_.predict(X_test_p)
print(X_test[0])
print(X_test_s[0])
print(X_test_p[0])
print(y_pred_cl[0])

print("\n===== Classical SVM Performance =====")
print(classification_report(y_test, y_pred_cl))

# =======================
# ⚛️ QUANTUM KERNEL SVM
# =======================
print("\nSetting up Quantum SVM...")
X_train_p =  X_train_p * np.pi
X_test_p = X_test_p * np.pi
'''
# Quantum Feature Map
feature_map = ZZFeatureMap(
    feature_dimension=pca.n_components_,
    reps=1,
    entanglement="linear"
)
'''
# Directly initialize FidelityQuantumKernel (no backend/sampler args)
import joblib
classifier = joblib.load('/content/drive/MyDrive/qsvc_trained_model.pkl')


y_pred_q = classifier.predict(X_test_p)

print("\n===== Quantum SVM Performance =====")
print(classification_report(y_test, y_pred_q))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_test, y_pred_q))


print(f"✅ Model saved successfully to: {save_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
import working properly
PCA Explained Variance: 1.000
[726.5048072   38.08393948  50.91595054]
[ 0.48019905  2.07331595 -0.74526644]
[-1.93267062 -0.77089705  0.86899612]
0

===== Classical SVM Performance =====
              precision    recall  f1-score   support

           0       1.00      0.67      0.80         3
           1       0.75      1.00      0.86         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6


Setting up Quantum SVM...


ModuleNotFoundError: No module named 'qiskit.circuit.quantumregister'

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/qsvc_trained_model.pkl'

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import joblib

# Quantum libraries
from qiskit_aer import Aer
from qiskit.circuit.library import ZZFeatureMap
from qiskit.circuit import QuantumCircuit,ParameterVector
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.optimizers import SPSA
from qiskit_machine_learning.kernels import TrainableFidelityQuantumKernel
from qiskit_machine_learning.kernels.algorithms import QuantumKernelTrainer

print("import working properly")
DATA_CSV = "data/sensor_readings4.csv"   # your dataset path
SOIL_COL = "Soil Moisture"              # feature column (example name)
TARGET_COL = "Pump Data"                # label column (0 = off, 1 = on)
SOIL_THRESHOLD = 0.25                   # threshold to generate labels if missing
PCA_COMPONENTS = 3                     # also number of qubits
TEST_SIZE = 0.3
RANDOM_STATE = 42
if not os.path.exists(DATA_CSV):
    raise FileNotFoundError(f"File not found: {DATA_CSV}")

df = pd.read_csv(DATA_CSV)
# Select numeric columns for training
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET_COL]
X = df[num_cols].values
y = df[TARGET_COL].values

# =======================
# 🔀 TRAIN-TEST SPLIT
# =======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

# =======================
# ⚗️ PREPROCESSING (Scaling + PCA)
# =======================
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

pca = PCA(n_components=min(PCA_COMPONENTS, X_train_s.shape[1]), random_state=RANDOM_STATE).fit(X_train_s)
X_train_p = pca.transform(X_train_s)
X_test_p = pca.transform(X_test_s)

print(f"PCA Explained Variance: {pca.explained_variance_ratio_.sum():.3f}")

# =======================
# 💡 CLASSICAL BASELINE (RBF SVM)
# =======================
clf = SVC(kernel='rbf', class_weight='balanced', random_state=RANDOM_STATE)
param_grid = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto']}
gs = GridSearchCV(clf, param_grid, cv=3, scoring='f1', n_jobs=-1)
gs.fit(X_train_p, y_train)
y_pred_cl = gs.best_estimator_.predict(X_test_p)

print("\n===== Classical SVM Performance =====")
print(classification_report(y_test, y_pred_cl))

# =======================
# ⚛️ QUANTUM KERNEL SVM
# =======================
print("\nSetting up Quantum SVM...")
X_train_p =  X_train_p * np.pi   # or 2*np.pi
X_test_p = X_test_p * np.pi

num_features = pca.n_components_
def build_trainable_feature_map(num_features):
    # Create base data-encoding map
    base_map = ZZFeatureMap(feature_dimension=num_features, reps=1, entanglement='full')

    # Add trainable rotation parameters
    theta = ParameterVector("θ", length=num_features)
    qc = QuantumCircuit(num_features)
    qc.compose(base_map, inplace=True)

    # Add trainable Ry gates after encoding
    for i in range(num_features):
        qc.ry(theta[i], i)

    return qc, theta

num_features = int(X_train_p.shape[1])
feature_map, theta = build_trainable_feature_map(num_features)

kernel = TrainableFidelityQuantumKernel(feature_map=feature_map, training_parameters=theta)
optimizer = SPSA(maxiter=20)

trainer = QuantumKernelTrainer(quantum_kernel=kernel, optimizer=optimizer, loss="svc_loss")
result = trainer.fit(X_train_p, y_train)

trained_kernel = result.quantum_kernel
classifier = QSVC(quantum_kernel=trained_kernel)
classifier.fit(X_train_p, y_train)
y_pred_q = classifier.predict(X_test_p)
print("Quantum SVM accuracy:", accuracy_score(y_test, y_pred_q))


import working properly
PCA Explained Variance: 1.000

===== Classical SVM Performance =====
              precision    recall  f1-score   support

           0       1.00      0.67      0.80         3
           1       0.75      1.00      0.86         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6


Setting up Quantum SVM...


/tmp/ipykernel_1091/1454904898.py:81: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  base_map = ZZFeatureMap(feature_dimension=num_features, reps=1, entanglement='full')
/usr/local/lib/python3.12/dist-packages/qiskit_machine_learning/optimizers/spsa.py:349: RuntimeWarning: divide by zero encountered in scalar divide
  a = target_magnitude / avg_magnitudes
/usr/local/lib/python3.12/dist-packages/qiskit_machine_learning/optimizers/spsa.py:571: RuntimeWarning: invalid value encountered in multiply
  update = update * next(eta)


AlgorithmError: 'Sampler job failed!'

In [ ]:
!pip install pennylane scikit-learn numpy matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 158.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


In [ ]:
# Function to compute the single kernel element K(x_a, x_b)
def quantum_kernel(x_a, x_b):
    # We take the probability of the all-zero state
    return kernel_circuit(x_a, x_b)[0]

In [ ]:
# Function to compute the full kernel matrix for two datasets
def kernel_matrix(A, B):
    """Computes the matrix K_ij = K(A_i, B_j)"""
    return np.array([[quantum_kernel(a, b) for b in B] for a in A])

# Example usage (assuming X_train and X_test are your data)
# K_train = kernel_matrix(X_train, X_train)
# K_test = kernel_matrix(X_test, X_train)
# Note: K_test computes similarity between test data and all *training* support vectors